### Model Tuning Procedure (Fixed Test + 5-Fold CV)

1. Build one fixed stratified `test.csv` (never used for tuning).
2. Put all remaining rows into `trainval.csv`.
3. Create 5 stratified folds from `trainval.csv` for tuning.
4. Tune with fold train/val files and average metrics across folds.
5. Retrain final model on all `trainval.csv` and evaluate once on `test.csv`.

In [1]:
from pathlib import Path
import importlib
import pandas as pd
import src.preprocess.split as split_module

# Reload to pick up latest edits when the notebook kernel already imported the module.
split_module = importlib.reload(split_module)
split_dataset = split_module.split_dataset

ANNOTATIONS_DIR = Path(r"C:\home\ben\REPSOL\Data\Annotations")
INPUT_CSV = ANNOTATIONS_DIR / "audio_annotations.csv"

# Use simple 70/15/15 split (no CV folds)
train_df, val_df, test_df = split_dataset(INPUT_CSV, ANNOTATIONS_DIR, seed=42)

print("\nCreated files:")
for rel in [
    "train.csv",
    "val.csv",
    "test.csv",
]:
    p = ANNOTATIONS_DIR / rel
    print(rel, "->", "OK" if p.exists() else "MISSING")

print("\nTest set class counts:")
print(test_df["category"].value_counts().sort_index())

# Optional sanity check: class distribution table
all_counts = pd.read_csv(INPUT_CSV)["category"].value_counts().sort_index().rename("all")
train_counts = train_df["category"].value_counts().sort_index().rename("train")
val_counts = val_df["category"].value_counts().sort_index().rename("val")
test_counts = test_df["category"].value_counts().sort_index().rename("test")

distribution = pd.concat([all_counts, train_counts, val_counts, test_counts], axis=1).fillna(0).astype(int)
print("\nClass distribution (all/train/val/test):")
print(distribution)

Train: 1383
Val:   296
Test:  297

Created files:
train.csv -> OK
val.csv -> OK
test.csv -> OK

Test set class counts:
category
0 ActividadBASE_NO pattern activity                                33
1 Pulses HAMMERING                                                  6
2 Marked cycles 3 segundos                                         20
3 Continuous activity & tone 3.15kHz_SPL ALTO                     106
4 Continuous activity & tone 3.15kHz_ SPL BAJO                     39
5 Blasts                                                           11
6 Machinery continuous activity                                    76
7 Works_ sirens and knocks en altas frecuencias RAFAGAS a 3.15      6
Name: count, dtype: int64

Class distribution (all/train/val/test):
                                                    all  train  val  test
category                                                                 
0 ActividadBASE_NO pattern activity                 225    158   34    33
1 Pulses HAMMERING    

## Generate Spectrograms from Images

Convert spectrogram PNG images from `D:\REPSOL_Classification\{class}\Espectrogramas` into `.pt` tensor files organized by train/val/test splits based on the annotation CSVs.

In [2]:
import subprocess
import sys

packages = ["torch", "librosa", "torchaudio", "scikit-learn", "pandas"]
for pkg in packages:
    try:
        __import__(pkg.replace("-", "_"))
        print(f"✓ {pkg} already installed")
    except ImportError:
        print(f"Installing {pkg}...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])
        print(f"✓ {pkg} installed")

✓ torch already installed
✓ librosa already installed
✓ torchaudio already installed
Installing scikit-learn...
✓ scikit-learn installed
✓ pandas already installed


In [4]:
# Verify spectrogram structure and count files
from pathlib import Path

# Where spectrograms are stored (train/val/test splits)
SPECTROGRAM_DIR = Path(r"C:\home\ben\REPSOL\Data\Spectrograms")
print(f"Spectrogram directory: {SPECTROGRAM_DIR}")
print(f"Exists: {SPECTROGRAM_DIR.exists()}")

# Count all available .pt files by split
print(f"\nSpectrograms by split:")
splits_counts = {}
for split in ["train", "val", "test"]:
    split_dir = SPECTROGRAM_DIR / split
    if split_dir.exists():
        count = sum(1 for _ in split_dir.rglob("*.pt"))
        splits_counts[split] = count
        print(f"  {split}: {count} .pt files")
    else:
        print(f"  {split}: directory not found")
        splits_counts[split] = 0

total = sum(splits_counts.values())
print(f"\nTotal: {total} .pt files")
print(f"\nExpected: train 1383, val 296, test 297")
print(f"Coverage: train {100*splits_counts['train']//1383}%, val {100*splits_counts['val']//296}%, test {100*splits_counts['test']//297}%")

Spectrogram directory: C:\home\ben\REPSOL\Data\Spectrograms
Exists: True

Spectrograms by split:
  train: 1383 .pt files
  val: 296 .pt files
  test: 297 .pt files

Total: 1976 .pt files

Expected: train 1383, val 296, test 297
Coverage: train 100%, val 100%, test 100%


In [5]:
from pathlib import Path

SPEC_ROOT = ANNOTATIONS_DIR.parent / "Spectrograms"

train_pts = list((SPEC_ROOT / "train").rglob("*.pt")) if (SPEC_ROOT / "train").exists() else []
val_pts = list((SPEC_ROOT / "val").rglob("*.pt")) if (SPEC_ROOT / "val").exists() else []
test_pts = list((SPEC_ROOT / "test").rglob("*.pt")) if (SPEC_ROOT / "test").exists() else []

total_pts = len(train_pts) + len(val_pts) + len(test_pts)

print(f"\n✓ Spectrograms Summary:")
print(f"  train: {len(train_pts)} .pt files")
print(f"  val:   {len(val_pts)} .pt files")
print(f"  test:  {len(test_pts)} .pt files")
print(f"  Total: {total_pts} .pt files")

# Show sample class structure
for split_name, pts_list in [("train", train_pts), ("val", val_pts), ("test", test_pts)]:
    if pts_list:
        print(f"\nSample {split_name} classes:")
        classes_seen = set()
        for p in pts_list[:5]:
            classes_seen.add(p.parent.name)
        for cls in sorted(classes_seen)[:2]:
            count = len(list((SPEC_ROOT / split_name / cls).glob("*.pt")))
            print(f"  {cls}: {count} files")


✓ Spectrograms Summary:
  train: 1383 .pt files
  val:   296 .pt files
  test:  297 .pt files
  Total: 1976 .pt files

Sample train classes:
  0 ActividadBASE_NO pattern activity: 158 files

Sample val classes:
  0 ActividadBASE_NO pattern activity: 34 files

Sample test classes:
  0 ActividadBASE_NO pattern activity: 33 files
